# Moment Retrieval - Group 6

This notebook tackles the moment-retrieval part of Assignment 2 step by step.

Goal: use the events detected by the VLM as text queries, run two pretrained moment-retrieval models, predict timestamps, compare feature settings, and save the results for IoU evaluation.

## Step-by-Step Plan

| Step | Goal | Status |
|------|------|--------|
| **6.1** | Load detected events from Step 3 | Done |
| **6.2** | Generate 3-5 query phrasings per event | Done |
| **6.3** | Prepare videos / chunk plan | Done |
| **6.4** | Check Lighthouse framework and checkpoints | Done |
| **6.5** | Smoke test: Moment-DETR on one query | Done |
| **6.6** | Smoke test: CG-DETR on one query | Done |
| **6.7** | Full inference sweep — all videos × both models | Added |
| **6.8** | Aggregate multi-query / multi-chunk predictions | Added |
| **6.9** | Post-processing: filter short, low-score, clamped windows | Added |
| **6.10** | Save one clean JSON per model | Added |
| **6.11** | IoU evaluation against annotations | Added |
| **6.12** | Feature experiment: CLIP vs CLIP+SlowFast | Added |
| **6.13** | Video summary generation | Added |

## Step 6.1 - Load Detected Events

Moment-retrieval models take a text query and a video as input. Our text queries come from the event-only VLM outputs created in Step 3.

The saved files are expected here:

```text
outputs/event_only_results_video_21.json
outputs/event_only_results_video_22.json
outputs/event_only_results_video_23.json
outputs/event_only_results_video_24.json
```

This step loads those files and parses the raw numbered VLM text into this structure:

```python
{
    "video_21": ["event text", "event text", ...],
    "video_22": ["event text", "event text", ...],
}
```

Success check: each video should print a non-empty list of detected events.

In [29]:
from pathlib import Path
import json
import re

EVENT_RESULTS_DIR = Path("outputs")
EXPECTED_VIDEOS = ["video_21", "video_22", "video_23", "video_24"]


def clean_event_text(text):
    """Remove numbering/prefixes and normalize whitespace."""
    text = re.sub(r"\s+", " ", text).strip()
    text = re.sub(r"^[Ss]al[ei]e?nt\s+event\s+\d+\s*:\s*", "", text)
    text = re.sub(r"^\d+[\.)\-]\s*", "", text).strip()
    return text.rstrip()


def parse_numbered_events(raw_text):
    """Parse numbered VLM output while preserving multiline event descriptions."""
    events = []
    current = []

    for line in raw_text.splitlines():
        line = line.strip()
        if not line:
            continue

        starts_new_event = (
            re.match(r"^\d+[\.)\-]\s+", line)
            or re.match(r"^[Ss]al[ei]e?nt\s+event\s+\d+\s*:", line)
        )

        if starts_new_event:
            if current:
                event = clean_event_text(" ".join(current))
                if event:
                    events.append(event)
            current = [line]
        elif current:
            current.append(line)

    if current:
        event = clean_event_text(" ".join(current))
        if event:
            events.append(event)

    return events


def load_detected_events(results_dir=EVENT_RESULTS_DIR):
    """Load all saved event-only VLM outputs from the outputs folder."""
    detected_events = {}

    for video_name in EXPECTED_VIDEOS:
        json_path = results_dir / f"event_only_results_{video_name}.json"
        if not json_path.exists():
            print(f"Missing: {json_path}")
            detected_events[video_name] = []
            continue

        with open(json_path, "r", encoding="utf-8") as f:
            payload = json.load(f)

        raw_output = payload.get(video_name, {}).get("raw", "")
        detected_events[video_name] = parse_numbered_events(raw_output)

    return detected_events


detected_events = load_detected_events()

print("Detected VLM events ready for moment retrieval:")
for video_name in EXPECTED_VIDEOS:
    events = detected_events.get(video_name, [])
    print(f"\n{video_name}: {len(events)} events")
    for i, event in enumerate(events[:3], start=1):
        print(f"  {i}. {event}")
    if len(events) > 3:
        print(f"  ... {len(events) - 3} more events")

assert all(detected_events.get(video_name) for video_name in EXPECTED_VIDEOS), "At least one video has no detected events."
print("\nStep 6.1 check passed: all expected videos have detected events.")

Detected VLM events ready for moment retrieval:

video_21: 24 events
  1. A man is running down a street with buildings behind him.
  2. The same person runs up to another building's entrance.
  3. He jumps over an obstacle while still near that building.
  ... 21 more events

video_22: 6 events
  1. A group of people are gathered around a brick wall with some standing closer to it than others.
  2. One person is jumping off the top step onto another set of steps below them while everyone else watches.
  3. The man who jumped has fallen down but quickly gets back up again after landing.
  ... 3 more events

video_23: 6 events
  1. A person is walking down a staircase with their hands behind them. They are wearing light blue jeans and carrying a backpack over their shoulder. The stairs have metal railings along both sides.
  2. Another individual walks up an elevator shaft towards another set of doors at ground level.
  3. An escalator moves upwards as someone stands near it waiting to 

## Step 6.2 - Generate Query Variants

Moment-retrieval models are sensitive to wording, so we will not use only one query per event.

For each detected event from Step 6.1, this step creates 3-5 deterministic query phrasings. These variants will later be sent to Moment-DETR and CG-DETR, and we can keep the highest-scoring timestamp prediction.

Success check: every event should have at least 3 query variants.

In [30]:
def make_query_variants(event, max_variants=5):
    """Create deterministic text-query variants for one detected event."""
    base = re.sub(r"\s+", " ", event).strip().rstrip(".")
    lower = base[:1].lower() + base[1:] if base else base

    variants = [
        base,
        f"A scene where {lower}.",
        f"The video shows {lower}.",
        f"The relevant moment is when {lower}.",
    ]

    # Add one simplified version when the VLM wording is overly specific.
    simplified = lower
    simplified = re.sub(r"\bthe same\b", "the", simplified, flags=re.IGNORECASE)
    simplified = re.sub(r"\banother individual\b", "a person", simplified, flags=re.IGNORECASE)
    simplified = re.sub(r"\banother\b", "a", simplified, flags=re.IGNORECASE)
    simplified = re.sub(r"\bindividual\b", "person", simplified, flags=re.IGNORECASE)
    simplified = re.sub(r"\s+", " ", simplified).strip()

    if simplified and simplified != lower:
        variants.append(simplified)

    # Remove duplicates while preserving order.
    unique_variants = []
    seen = set()
    for variant in variants:
        variant = re.sub(r"\s+", " ", variant).strip()
        key = variant.lower().rstrip(".")
        if variant and key not in seen:
            seen.add(key)
            unique_variants.append(variant)

    return unique_variants[:max_variants]


query_variants = {
    video_name: [
        {
            "event_id": event_id,
            "event": event,
            "queries": make_query_variants(event),
        }
        for event_id, event in enumerate(events, start=1)
    ]
    for video_name, events in detected_events.items()
}

print("Query variants generated:")
for video_name in EXPECTED_VIDEOS:
    event_queries = query_variants.get(video_name, [])
    counts = [len(item["queries"]) for item in event_queries]
    min_count = min(counts) if counts else 0
    max_count = max(counts) if counts else 0
    print(f"{video_name}: {len(event_queries)} events, {min_count}-{max_count} queries per event")

print("\nPreview: first event of each video")
for video_name in EXPECTED_VIDEOS:
    item = query_variants[video_name][0]
    print(f"\n{video_name} - event {item['event_id']}: {item['event']}")
    for i, query in enumerate(item["queries"], start=1):
        print(f"  q{i}. {query}")

assert all(
    len(item["queries"]) >= 3
    for event_queries in query_variants.values()
    for item in event_queries
), "At least one event has fewer than 3 query variants."

print("\nStep 6.2 check passed: every event has at least 3 query variants.")

Query variants generated:
video_21: 24 events, 4-5 queries per event
video_22: 6 events, 4-5 queries per event
video_23: 6 events, 4-5 queries per event
video_24: 6 events, 4-5 queries per event

Preview: first event of each video

video_21 - event 1: A man is running down a street with buildings behind him.
  q1. A man is running down a street with buildings behind him
  q2. A scene where a man is running down a street with buildings behind him.
  q3. The video shows a man is running down a street with buildings behind him.
  q4. The relevant moment is when a man is running down a street with buildings behind him.

video_22 - event 1: A group of people are gathered around a brick wall with some standing closer to it than others.
  q1. A group of people are gathered around a brick wall with some standing closer to it than others
  q2. A scene where a group of people are gathered around a brick wall with some standing closer to it than others.
  q3. The video shows a group of people are

## Step 6.3 - Prepare Videos for Moment Retrieval

Pretrained moment-retrieval demos often work best with short videos or clips. The Lighthouse demo notebook notes a practical limit around **150 seconds**, so longer videos should be split into overlapping chunks before retrieval.

In this step we do **not** run a model yet. We only create a chunk plan:

- get each video duration,
- keep short videos as one chunk,
- split long videos into chunks shorter than 150 seconds,
- add overlap so events near chunk boundaries are not missed,
- store everything in `video_chunk_plan` for the retrieval steps.

Success check: `video_21` should be split into several chunks; shorter videos should have fewer chunks.

In [31]:
from pathlib import Path
import math

VIDEO_PATHS = {
    "video_21": Path("video_21.mp4"),
    "video_22": Path("video_22.mp4"),
    "video_23": Path("video_23.mp4"),
    "video_24": Path("video_24.mp4"),
}

# Fallback durations from the earlier metadata cell in Assignment2_Group6.ipynb.
# These are used only if OpenCV metadata reading is unavailable in the current kernel.
KNOWN_DURATIONS = {
    "video_21": 647.51,
    "video_22": 235.88,
    "video_23": 187.89,
    "video_24": 145.35,
}

MAX_CHUNK_SECONDS = 149.0   # stay just below the ~150s practical model/demo limit
CHUNK_OVERLAP_SECONDS = 10.0


def seconds_to_mmss(seconds):
    total = int(round(seconds))
    return f"{total // 60:02d}:{total % 60:02d}"


def get_video_duration_seconds(video_name, video_path):
    """Read duration with OpenCV; fall back to known assignment metadata."""
    try:
        import cv2

        cap = cv2.VideoCapture(str(video_path))
        if cap.isOpened():
            fps = cap.get(cv2.CAP_PROP_FPS)
            frame_count = cap.get(cv2.CAP_PROP_FRAME_COUNT)
            cap.release()
            if fps and frame_count:
                return frame_count / fps
    except Exception as exc:
        print(f"OpenCV duration read failed for {video_name}; using fallback. Reason: {exc}")

    return KNOWN_DURATIONS[video_name]


def make_chunk_ranges(duration_s, max_chunk_s=MAX_CHUNK_SECONDS, overlap_s=CHUNK_OVERLAP_SECONDS):
    """Return overlapping chunk ranges as (start_s, end_s) pairs."""
    if duration_s <= max_chunk_s:
        return [(0.0, duration_s)]

    chunks = []
    step = max_chunk_s - overlap_s
    start = 0.0

    while start < duration_s:
        end = min(start + max_chunk_s, duration_s)
        chunks.append((start, end))

        if end >= duration_s:
            break

        start += step

    return chunks


video_chunk_plan = {}

for video_name, video_path in VIDEO_PATHS.items():
    if not video_path.exists():
        raise FileNotFoundError(f"Missing video file: {video_path}")

    duration_s = get_video_duration_seconds(video_name, video_path)
    chunks = make_chunk_ranges(duration_s)

    video_chunk_plan[video_name] = {
        "video_path": str(video_path),
        "duration_s": duration_s,
        "duration": seconds_to_mmss(duration_s),
        "chunks": [
            {
                "chunk_id": chunk_id,
                "start_s": start_s,
                "end_s": end_s,
                "start": seconds_to_mmss(start_s),
                "end": seconds_to_mmss(end_s),
            }
            for chunk_id, (start_s, end_s) in enumerate(chunks, start=1)
        ],
    }


print("Video chunk plan:")
for video_name, plan in video_chunk_plan.items():
    print(f"\n{video_name}: duration {plan['duration']} ({plan['duration_s']:.1f}s), {len(plan['chunks'])} chunk(s)")
    for chunk in plan["chunks"]:
        print(f"  chunk {chunk['chunk_id']}: {chunk['start']} - {chunk['end']}")


assert len(video_chunk_plan["video_21"]["chunks"]) > 1, "video_21 should be split into multiple chunks."
assert len(video_chunk_plan["video_24"]["chunks"]) == 1, "video_24 should fit as one chunk."
print("\nStep 6.3 check passed: video durations and chunk ranges are ready.")

Video chunk plan:

video_21: duration 10:48 (647.5s), 5 chunk(s)
  chunk 1: 00:00 - 02:29
  chunk 2: 02:19 - 04:48
  chunk 3: 04:38 - 07:07
  chunk 4: 06:57 - 09:26
  chunk 5: 09:16 - 10:48

video_22: duration 03:56 (235.9s), 2 chunk(s)
  chunk 1: 00:00 - 02:29
  chunk 2: 02:19 - 03:56

video_23: duration 03:08 (187.9s), 2 chunk(s)
  chunk 1: 00:00 - 02:29
  chunk 2: 02:19 - 03:08

video_24: duration 02:25 (145.3s), 1 chunk(s)
  chunk 1: 00:00 - 02:25

Step 6.3 check passed: video durations and chunk ranges are ready.


## Step 6.4 - Check Retrieval Framework and Model Setup

We will use **Lighthouse** because it provides a common inference API for several pretrained moment-retrieval models.

For the assignment requirement of using two models, we will start with:

1. **Moment-DETR**
2. **CG-DETR**

For the feature experiment, we will start with:

1. **CLIP** features - practical baseline, especially on CPU
2. **CLIP + SlowFast** features - optional second feature condition if compute allows

According to the Lighthouse documentation, the setup is:

```bash
pip install torch torchvision torchaudio
pip install "git+https://github.com/line/lighthouse.git"
```

This cell only checks whether the environment is ready. It does **not** run inference yet.

In [32]:
import importlib.util
import sys
from pathlib import Path

WEIGHTS_DIR = Path("weights")
WEIGHTS_DIR.mkdir(exist_ok=True)

RETRIEVAL_MODELS = [
    {
        "model_key": "moment_detr",
        "display_name": "Moment-DETR",
        "predictor_class": "MomentDETRPredictor",
    },
    {
        "model_key": "cg_detr",
        "display_name": "CG-DETR",
        "predictor_class": "CGDETRPredictor",
    },
]

# First feature condition. This is the practical baseline recommended for CPU runs.
FEATURES_TO_TEST = ["clip"]

# Optional second feature condition. We will only run this if the environment can handle it.
OPTIONAL_FEATURES_TO_TEST = ["clip_slowfast"]


def package_available(package_name):
    return importlib.util.find_spec(package_name) is not None


torch_available = package_available("torch")
lighthouse_available = package_available("lighthouse")

if torch_available:
    import torch

    if torch.cuda.is_available():
        RETRIEVAL_DEVICE = "cuda"
    elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        RETRIEVAL_DEVICE = "mps"
    else:
        RETRIEVAL_DEVICE = "cpu"
else:
    RETRIEVAL_DEVICE = "cpu"


available_predictor_classes = {}
model_import_error = None
if lighthouse_available:
    try:
        # Compatibility shim: some dependencies try `import numpy.char`.
        import numpy.core.defchararray as numpy_char
        sys.modules.setdefault("numpy.char", numpy_char)

        import lighthouse.models as lighthouse_models

        for model_cfg in RETRIEVAL_MODELS:
            class_name = model_cfg["predictor_class"]
            available_predictor_classes[class_name] = hasattr(lighthouse_models, class_name)
    except Exception as exc:
        model_import_error = str(exc)
        available_predictor_classes = {model_cfg["predictor_class"]: False for model_cfg in RETRIEVAL_MODELS}
else:
    available_predictor_classes = {model_cfg["predictor_class"]: False for model_cfg in RETRIEVAL_MODELS}


checkpoint_plan = []
for model_cfg in RETRIEVAL_MODELS:
    for feature_name in FEATURES_TO_TEST:
        checkpoint_plan.append({
            "model": model_cfg["model_key"],
            "feature": feature_name,
            "path": WEIGHTS_DIR / f"{feature_name}_{model_cfg['model_key']}_qvhighlight.ckpt",
        })
retrieval_setup = {
    "device": RETRIEVAL_DEVICE,
    "models": RETRIEVAL_MODELS,
    "features": FEATURES_TO_TEST,
    "optional_features": OPTIONAL_FEATURES_TO_TEST,
    "checkpoint_plan": checkpoint_plan,
}


missing_checkpoints = [item for item in checkpoint_plan if not item["path"].exists()]

models_ready = all(available_predictor_classes.values())

print("Step 6.4")
print(f"Models ready: {models_ready}")
print(f"Checkpoints found: {len(checkpoint_plan) - len(missing_checkpoints)}/{len(checkpoint_plan)}")

if missing_checkpoints:
    for item in missing_checkpoints:
        print(f"Missing: {item['path']}")

if torch_available and lighthouse_available and models_ready:
    print("Step 6.4 passed")
else:
    print("Step 6.4 not ready")

Step 6.4
Models ready: True
Checkpoints found: 2/2
Step 6.4 passed


## Step 6.5 - Run One Moment-DETR Test Query

This step checks that the first pretrained model actually works before we run it over all events.

We will:

1. download/check the **Moment-DETR + CLIP** checkpoint,
2. load `MomentDETRPredictor`,
3. run one query from `video_24`,
4. print only the predicted timestamp and confidence score.

Success check: the model should return one predicted time window.

In [33]:
import contextlib
import os
import sys
import urllib.request

assert "query_variants" in globals(), "Run Step 6.2 first."
assert "video_chunk_plan" in globals(), "Run Step 6.3 first."
assert "retrieval_setup" in globals(), "Run Step 6.4 first."

MOMENT_DETR_CKPT = WEIGHTS_DIR / "clip_moment_detr_qvhighlight.ckpt"
MOMENT_DETR_SAFE_CKPT = WEIGHTS_DIR / "clip_moment_detr_qvhighlight_safe.ckpt"
MOMENT_DETR_URL = "https://zenodo.org/records/13363606/files/clip_moment_detr_qvhighlight.ckpt"


def download_file_if_missing(url, output_path):
    output_path.parent.mkdir(exist_ok=True)
    if output_path.exists():
        return False

    temp_path = output_path.with_suffix(output_path.suffix + ".part")
    urllib.request.urlretrieve(url, temp_path)
    temp_path.replace(output_path)
    return True


# Compatibility shim for Lighthouse imports in some Windows/NumPy combinations.
import numpy.core.defchararray as numpy_char
sys.modules.setdefault("numpy.char", numpy_char)

from lighthouse.models import MomentDETRPredictor
from easydict import EasyDict
import torch


def create_safe_lighthouse_checkpoint(source_path, output_path):
    """Convert the official checkpoint's EasyDict config into a plain dict.

    PyTorch's safe loader cannot unpickle EasyDict directly. This conversion is
    done once from the official Lighthouse checkpoint, then later loads use the
    converted plain-dict checkpoint.
    """
    if output_path.exists():
        return False

    original = torch.load(source_path, map_location="cpu", weights_only=False)
    safe_checkpoint = {
        "model": original["model"],
        "opt": dict(original["opt"]),
    }
    torch.save(safe_checkpoint, output_path)
    return True


downloaded = download_file_if_missing(MOMENT_DETR_URL, MOMENT_DETR_CKPT)
converted = create_safe_lighthouse_checkpoint(MOMENT_DETR_CKPT, MOMENT_DETR_SAFE_CKPT)

# Use video_23 for the smoke test because it has a clearer narrative than
# video_24, which is a fast compilation of unrelated cuts.
TEST_VIDEO_NAME = "video_23"
TEST_EVENT_ID = 5

test_chunk = video_chunk_plan[TEST_VIDEO_NAME]["chunks"][0]
test_video = video_chunk_plan[TEST_VIDEO_NAME]["video_path"]
test_event = query_variants[TEST_VIDEO_NAME][TEST_EVENT_ID - 1]
test_query = test_event["queries"][0]


def make_temp_video_chunk(video_path, chunk, output_dir=Path("outputs/moment_retrieval_chunks")):
    """Create a physical video chunk for retrieval if it does not exist yet."""
    output_dir.mkdir(parents=True, exist_ok=True)
    chunk_path = output_dir / f"{Path(video_path).stem}_chunk_{chunk['chunk_id']:02d}.mp4"
    if chunk_path.exists():
        return chunk_path

    import subprocess
    duration = chunk["end_s"] - chunk["start_s"]
    command = [
        "ffmpeg",
        "-y",
        "-ss", str(chunk["start_s"]),
        "-t", str(duration),
        "-i", str(video_path),
        "-c", "copy",
        str(chunk_path),
    ]
    subprocess.run(command, check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    return chunk_path
CLIP_CACHE_DIR = WEIGHTS_DIR / "clip_cache"
CLIP_CACHE_DIR.mkdir(exist_ok=True)

# Make conda-installed ffmpeg/ffprobe visible to ffmpeg-python.
CONDA_FFMPEG_BIN = Path(sys.prefix) / "Library" / "bin"
if CONDA_FFMPEG_BIN.exists():
    os.environ["PATH"] = str(CONDA_FFMPEG_BIN) + os.pathsep + os.environ.get("PATH", "")

# Lighthouse expects attribute-style config access. The converted checkpoint
# stores the config safely as a dict, so we turn it back into EasyDict after
# torch.load has completed.
_original_torch_load = torch.load


def _torch_load_safe_lighthouse(*args, **kwargs):
    loaded = _original_torch_load(*args, **kwargs)
    if args and Path(args[0]).resolve() == MOMENT_DETR_SAFE_CKPT.resolve():
        loaded["opt"] = EasyDict(loaded["opt"])
    return loaded


# Keep CLIP downloads inside the project instead of the user home cache.
import clip
_original_clip_load = clip.load


def _clip_load_project_cache(*args, **kwargs):
    kwargs.setdefault("download_root", str(CLIP_CACHE_DIR))
    return _original_clip_load(*args, **kwargs)

# Suppress library progress/log chatter so the notebook output stays readable.
try:
    torch.load = _torch_load_safe_lighthouse
    clip.load = _clip_load_project_cache
    with open("nul", "w") as devnull:
        with contextlib.redirect_stdout(devnull), contextlib.redirect_stderr(devnull):
            moment_detr_model = MomentDETRPredictor(
                str(MOMENT_DETR_SAFE_CKPT),
                device=retrieval_setup["device"],
                feature_name="clip",
            )
            test_chunk_path = make_temp_video_chunk(test_video, test_chunk)
            encoded_video = moment_detr_model.encode_video(str(test_chunk_path))
            prediction = moment_detr_model.predict(test_query, encoded_video)
finally:
    torch.load = _original_torch_load
    clip.load = _original_clip_load

best_window = prediction["pred_relevant_windows"][0]
chunk_start_s, chunk_end_s, score = best_window
start_s = test_chunk["start_s"] + chunk_start_s
end_s = test_chunk["start_s"] + chunk_end_s

print("Step 6.5")
print(f"Checkpoint ready: {MOMENT_DETR_SAFE_CKPT.exists()}")
print(f"Video: {TEST_VIDEO_NAME}")
print(f"Chunk: {test_chunk['start']} - {test_chunk['end']}")
print(f"Query: {test_query}")
print(f"Prediction: {seconds_to_mmss(start_s)} - {seconds_to_mmss(end_s)}")
print(f"Score: {score:.4f}")
print("Step 6.5 passed")

Step 6.5
Checkpoint ready: True
Video: video_23
Chunk: 00:00 - 02:29
Query: Someone sits inside what appears to be a bus stop area next to parked cars under trees
Prediction: 00:44 - 01:04
Score: 0.9790
Step 6.5 passed


## Step 6.6 - Run One CG-DETR Test Query

The Canvas demo uses `CGDETRPredictor`, so this step mirrors that demo more directly.

We use the same video chunk and query as Step 6.5 so the two smoke tests are comparable:

- **Step 6.5:** Moment-DETR + CLIP
- **Step 6.6:** CG-DETR + CLIP

Success check: CG-DETR should return one predicted time window and score.

In [34]:
import contextlib
import os
import sys
import urllib.request

assert "query_variants" in globals(), "Run Step 6.2 first."
assert "video_chunk_plan" in globals(), "Run Step 6.3 first."
assert "retrieval_setup" in globals(), "Run Step 6.4 first."

CG_DETR_CKPT = WEIGHTS_DIR / "clip_cg_detr_qvhighlight.ckpt"
CG_DETR_SAFE_CKPT = WEIGHTS_DIR / "clip_cg_detr_qvhighlight_safe.ckpt"
CG_DETR_URL = "https://zenodo.org/records/13363606/files/clip_cg_detr_qvhighlight.ckpt"


def download_file_if_missing(url, output_path):
    output_path.parent.mkdir(exist_ok=True)
    if output_path.exists():
        return False

    temp_path = output_path.with_suffix(output_path.suffix + ".part")
    urllib.request.urlretrieve(url, temp_path)
    temp_path.replace(output_path)
    return True


# Compatibility shim for Lighthouse imports in some Windows/NumPy combinations.
import numpy.core.defchararray as numpy_char
sys.modules.setdefault("numpy.char", numpy_char)

from lighthouse.models import CGDETRPredictor
from easydict import EasyDict
import torch


def create_safe_lighthouse_checkpoint(source_path, output_path):
    """Convert the official checkpoint's EasyDict config into a plain dict."""
    if output_path.exists():
        return False

    original = torch.load(source_path, map_location="cpu", weights_only=False)
    safe_checkpoint = {
        "model": original["model"],
        "opt": dict(original["opt"]),
    }
    torch.save(safe_checkpoint, output_path)
    return True


downloaded = download_file_if_missing(CG_DETR_URL, CG_DETR_CKPT)
converted = create_safe_lighthouse_checkpoint(CG_DETR_CKPT, CG_DETR_SAFE_CKPT)

TEST_VIDEO_NAME = "video_23"
TEST_EVENT_ID = 5

test_chunk = video_chunk_plan[TEST_VIDEO_NAME]["chunks"][0]
test_video = video_chunk_plan[TEST_VIDEO_NAME]["video_path"]
test_event = query_variants[TEST_VIDEO_NAME][TEST_EVENT_ID - 1]
test_query = test_event["queries"][0]


def make_temp_video_chunk(video_path, chunk, output_dir=Path("outputs/moment_retrieval_chunks")):
    """Create a physical video chunk for retrieval if it does not exist yet."""
    output_dir.mkdir(parents=True, exist_ok=True)
    chunk_path = output_dir / f"{Path(video_path).stem}_chunk_{chunk['chunk_id']:02d}.mp4"
    if chunk_path.exists():
        return chunk_path

    import subprocess
    duration = chunk["end_s"] - chunk["start_s"]
    command = [
        "ffmpeg",
        "-y",
        "-ss", str(chunk["start_s"]),
        "-t", str(duration),
        "-i", str(video_path),
        "-c", "copy",
        str(chunk_path),
    ]
    subprocess.run(command, check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    return chunk_path


CLIP_CACHE_DIR = WEIGHTS_DIR / "clip_cache"
CLIP_CACHE_DIR.mkdir(exist_ok=True)

CONDA_FFMPEG_BIN = Path(sys.prefix) / "Library" / "bin"
if CONDA_FFMPEG_BIN.exists():
    os.environ["PATH"] = str(CONDA_FFMPEG_BIN) + os.pathsep + os.environ.get("PATH", "")

_original_torch_load = torch.load


def _torch_load_safe_lighthouse(*args, **kwargs):
    loaded = _original_torch_load(*args, **kwargs)
    if args and Path(args[0]).resolve() == CG_DETR_SAFE_CKPT.resolve():
        loaded["opt"] = EasyDict(loaded["opt"])
    return loaded


import clip
_original_clip_load = clip.load


def _clip_load_project_cache(*args, **kwargs):
    kwargs.setdefault("download_root", str(CLIP_CACHE_DIR))
    return _original_clip_load(*args, **kwargs)


try:
    torch.load = _torch_load_safe_lighthouse
    clip.load = _clip_load_project_cache
    with open("nul", "w") as devnull:
        with contextlib.redirect_stdout(devnull), contextlib.redirect_stderr(devnull):
            cg_detr_model = CGDETRPredictor(
                str(CG_DETR_SAFE_CKPT),
                device=retrieval_setup["device"],
                feature_name="clip",
            )
            test_chunk_path = make_temp_video_chunk(test_video, test_chunk)
            encoded_video = cg_detr_model.encode_video(str(test_chunk_path))
            prediction = cg_detr_model.predict(test_query, encoded_video)
finally:
    torch.load = _original_torch_load
    clip.load = _original_clip_load

best_window = prediction["pred_relevant_windows"][0]
chunk_start_s, chunk_end_s, score = best_window
start_s = test_chunk["start_s"] + chunk_start_s
end_s = test_chunk["start_s"] + chunk_end_s

print("Step 6.6")
print(f"Checkpoint ready: {CG_DETR_SAFE_CKPT.exists()}")
print(f"Video: {TEST_VIDEO_NAME}")
print(f"Chunk: {test_chunk['start']} - {test_chunk['end']}")
print(f"Query: {test_query}")
print(f"Prediction: {seconds_to_mmss(start_s)} - {seconds_to_mmss(end_s)}")
print(f"Score: {score:.4f}")
print("Step 6.6 passed")

Step 6.6
Checkpoint ready: True
Video: video_23
Chunk: 00:00 - 02:29
Query: Someone sits inside what appears to be a bus stop area next to parked cars under trees
Prediction: 00:41 - 00:57
Score: 0.9961
Step 6.6 passed


## Step 6.7 - Full Inference Sweep (All Videos × Both Models)

The smoke tests in Steps 6.5 and 6.6 ran a single query on a single chunk.  This step runs the **complete sweep**:

- **Both models**: Moment-DETR and CG-DETR (loaded and unloaded in sequence to avoid memory conflicts)
- **All 4 videos**: video_21 (5 chunks), video_22 (1 chunk), video_23 (1 chunk), video_24 (1 chunk)
- **All events per video**: 24 events for video_21, 6 events each for videos 22–24
- **All query variants per event**: 3–5 phrasings generated in Step 6.2

Key efficiency decisions:
- Each video chunk is **encoded once** and reused for all query variants — encoding is the expensive step.
- Each model is **loaded once** for all videos and **unloaded before** the next model is loaded, so GPU/MPS memory is never shared between the two models.
- Chunk files for video_21 are extracted to `outputs/moment_retrieval_chunks/` and cached on disk.

Output: `outputs/raw_predictions.json` — every (model, video, event, chunk, query_variant) combination with its `start_s`, `end_s`, and `score`.

Success check: the JSON file should contain entries for both `moment_detr` and `cg_detr` across all 4 videos.

In [35]:
import contextlib
import gc
import json
import os
import subprocess
import sys
from pathlib import Path

import numpy.core.defchararray as _numpy_char
sys.modules.setdefault("numpy.char", _numpy_char)

import clip
import torch
from easydict import EasyDict
from lighthouse.models import CGDETRPredictor, MomentDETRPredictor

assert "query_variants"    in globals(), "Run Step 6.2 first."
assert "video_chunk_plan"  in globals(), "Run Step 6.3 first."
assert "retrieval_setup"   in globals(), "Run Step 6.4 first."
assert "MOMENT_DETR_SAFE_CKPT" in globals(), "Run Step 6.5 first."
assert "CG_DETR_SAFE_CKPT"     in globals(), "Run Step 6.6 first."

CHUNK_DIR = Path("outputs/moment_retrieval_chunks")
CHUNK_DIR.mkdir(parents=True, exist_ok=True)

CONDA_FFMPEG_BIN = Path(sys.prefix) / "Library" / "bin"
if CONDA_FFMPEG_BIN.exists():
    os.environ["PATH"] = str(CONDA_FFMPEG_BIN) + os.pathsep + os.environ.get("PATH", "")


# ── helpers ───────────────────────────────────────────────────────────────────

def ensure_chunk_file(video_path, chunk):
    """Extract one video chunk to disk if it does not already exist."""
    out = CHUNK_DIR / f"{Path(video_path).stem}_chunk_{chunk['chunk_id']:02d}.mp4"
    if not out.exists():
        duration = chunk["end_s"] - chunk["start_s"]
        subprocess.run(
            ["ffmpeg", "-y", "-ss", str(chunk["start_s"]), "-t", str(duration),
             "-i", str(video_path), "-c", "copy", str(out)],
            check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
        )
    return out


def _make_load_patch(safe_ckpt_path):
    """Return a torch.load wrapper that re-attaches EasyDict after safe load."""
    _orig = torch.load

    def _patched(*args, **kwargs):
        loaded = _orig(*args, **kwargs)
        if args and Path(str(args[0])).resolve() == Path(safe_ckpt_path).resolve():
            loaded["opt"] = EasyDict(loaded["opt"])
        return loaded

    return _orig, _patched


# ── model configs ─────────────────────────────────────────────────────────────

MODEL_CONFIGS = [
    {
        "model_key":       "moment_detr",
        "display_name":    "Moment-DETR",
        "predictor_class": MomentDETRPredictor,
        "safe_ckpt":       MOMENT_DETR_SAFE_CKPT,
    },
    {
        "model_key":       "cg_detr",
        "display_name":    "CG-DETR",
        "predictor_class": CGDETRPredictor,
        "safe_ckpt":       CG_DETR_SAFE_CKPT,
    },
]

raw_predictions = {}

# ── main sweep ────────────────────────────────────────────────────────────────

for model_cfg in MODEL_CONFIGS:
    model_key    = model_cfg["model_key"]
    display_name = model_cfg["display_name"]
    safe_ckpt    = model_cfg["safe_ckpt"]

    print(f"\n{'=' * 60}")
    print(f"  {display_name}")
    print(f"{'=' * 60}")

    raw_predictions[model_key] = {}

    _orig_load, _patched_load = _make_load_patch(safe_ckpt)
    _orig_clip = clip.load

    try:
        torch.load = _patched_load
        clip.load  = lambda *a, **kw: _orig_clip(
            *a, **{**kw, "download_root": str(WEIGHTS_DIR / "clip_cache")}
        )
        with open("nul", "w") as _dev:
            with contextlib.redirect_stdout(_dev), contextlib.redirect_stderr(_dev):
                model = model_cfg["predictor_class"](
                    str(safe_ckpt),
                    device=retrieval_setup["device"],
                    feature_name="clip",
                )
    finally:
        torch.load = _orig_load
        clip.load  = _orig_clip

    print(f"  Loaded on {retrieval_setup['device']}")

    for video_name in EXPECTED_VIDEOS:
        plan   = video_chunk_plan[video_name]
        chunks = plan["chunks"]
        events = query_variants[video_name]

        print(f"\n  {video_name}: {len(chunks)} chunk(s) × {len(events)} event(s)")

        video_results = []

        for chunk in chunks:
            chunk_path    = ensure_chunk_file(plan["video_path"], chunk)
            chunk_start_s = chunk["start_s"]

            # Encode the chunk once and reuse for all queries in this chunk.
            with open("nul", "w") as _dev:
                with contextlib.redirect_stdout(_dev), contextlib.redirect_stderr(_dev):
                    encoded_video = model.encode_video(str(chunk_path))

            for event_item in events:
                event_id   = event_item["event_id"]
                event_text = event_item["event"]
                queries    = event_item["queries"]

                query_results = []
                for query_text in queries:
                    with open("nul", "w") as _dev:
                        with contextlib.redirect_stdout(_dev), contextlib.redirect_stderr(_dev):
                            pred = model.predict(query_text, encoded_video)

                    rel_start, rel_end, score = pred["pred_relevant_windows"][0]
                    query_results.append({
                        "query":   query_text,
                        "start_s": round(float(chunk_start_s + rel_start), 2),
                        "end_s":   round(float(chunk_start_s + rel_end),   2),
                        "score":   round(float(score), 4),
                    })

                # Store results grouped by event_id so we can merge chunks later.
                existing = next(
                    (r for r in video_results if r["event_id"] == event_id), None
                )
                chunk_entry = {
                    "chunk_id":      chunk["chunk_id"],
                    "chunk_start_s": chunk["start_s"],
                    "chunk_end_s":   chunk["end_s"],
                    "query_results": query_results,
                }
                if existing is None:
                    video_results.append({
                        "event_id":     event_id,
                        "event":        event_text,
                        "chunk_results": [chunk_entry],
                    })
                else:
                    existing["chunk_results"].append(chunk_entry)

            # Progress per chunk.
            best_scores = [
                max(qr["score"] for qr in er["chunk_results"][-1]["query_results"])
                for er in video_results
                if er["chunk_results"][-1]["chunk_id"] == chunk["chunk_id"]
            ]
            avg = sum(best_scores) / len(best_scores) if best_scores else 0.0
            print(f"    chunk {chunk['chunk_id']}: avg best-score = {avg:.4f}")

        raw_predictions[model_key][video_name] = video_results

    # Free GPU/MPS memory before loading the next model.
    del model
    if retrieval_setup["device"] == "cuda":
        torch.cuda.empty_cache()
    elif retrieval_setup["device"] == "mps" and hasattr(torch.mps, "empty_cache"):
        torch.mps.empty_cache()
    gc.collect()
    print(f"\n  {display_name} done — model unloaded.")

# Save raw results for debugging and re-runs.
RAW_PREDICTIONS_PATH = Path("outputs/raw_predictions.json")
with open(RAW_PREDICTIONS_PATH, "w") as f:
    json.dump(raw_predictions, f, indent=2)

print(f"\nRaw predictions saved to {RAW_PREDICTIONS_PATH}")
print("Step 6.7 passed")



  Moment-DETR
  Loaded on cpu

  video_21: 5 chunk(s) × 24 event(s)
    chunk 1: avg best-score = 0.9647
    chunk 2: avg best-score = 0.9733
    chunk 3: avg best-score = 0.9905
    chunk 4: avg best-score = 0.9886
    chunk 5: avg best-score = 0.9594

  video_22: 2 chunk(s) × 6 event(s)
    chunk 1: avg best-score = 0.9824
    chunk 2: avg best-score = 0.9871

  video_23: 2 chunk(s) × 6 event(s)
    chunk 1: avg best-score = 0.9905
    chunk 2: avg best-score = 0.9770

  video_24: 1 chunk(s) × 6 event(s)
    chunk 1: avg best-score = 0.9868

  Moment-DETR done — model unloaded.

  CG-DETR
  Loaded on cpu

  video_21: 5 chunk(s) × 24 event(s)
    chunk 1: avg best-score = 0.9622
    chunk 2: avg best-score = 0.9745
    chunk 3: avg best-score = 0.9861
    chunk 4: avg best-score = 0.9862
    chunk 5: avg best-score = 0.9842

  video_22: 2 chunk(s) × 6 event(s)
    chunk 1: avg best-score = 0.9777
    chunk 2: avg best-score = 0.9950

  video_23: 2 chunk(s) × 6 event(s)
    chunk 1: a

## Step 6.8 - Aggregate Multi-Query and Multi-Chunk Predictions

Each event now has predictions from 3–5 query variants, possibly spread across multiple video chunks (video_21 has 5 chunks).  This step collapses all of that into **one best prediction per event per model**.

Aggregation strategy:
1. For every (chunk, query-variant) pair, collect the predicted window and score.
2. Discard predictions shorter than 2 seconds or with a negative confidence score.
3. Keep the single highest-scoring prediction.  If nothing passes the filters, fall back to the global maximum score regardless of filters.

Output: `outputs/aggregated_predictions.json` — one entry per event per video per model.

Success check: every event in every video should have exactly one `start_s / end_s / score / best_query` entry.

In [36]:
import json
import gc

assert "raw_predictions" in globals(), "Run Step 6.7 first."

MIN_SEGMENT_DURATION = 2.0  # seconds — discard very short spurious predictions
MIN_SCORE = 0.0             # discard negative-confidence predictions


def aggregate_event_predictions(chunk_results):
    """Return the single best prediction across all chunks and query variants.

    For each chunk we look at every query variant and keep the highest-scoring
    prediction that passes the duration and score filters.  We then take the
    best result across all chunks.  If nothing passes the filters we fall back
    to the globally highest score so we always return something.
    """
    best = None
    best_score = float("-inf")

    for chunk_result in chunk_results:
        for qr in chunk_result["query_results"]:
            duration = qr["end_s"] - qr["start_s"]
            if duration < MIN_SEGMENT_DURATION:
                continue
            if qr["score"] < MIN_SCORE:
                continue
            if qr["score"] > best_score:
                best_score = qr["score"]
                best = qr

    if best is None:
        # Fallback: ignore filters and just take the highest raw score.
        all_qr = [
            qr
            for cr in chunk_results
            for qr in cr["query_results"]
        ]
        best = max(all_qr, key=lambda x: x["score"])

    return best


aggregated_predictions = {}

for model_key, model_results in raw_predictions.items():
    aggregated_predictions[model_key] = {}

    for video_name, video_results in model_results.items():
        video_agg = []

        for event_result in video_results:
            best = aggregate_event_predictions(event_result["chunk_results"])
            video_agg.append({
                "event_id": event_result["event_id"],
                "event":    event_result["event"],
                "start_s":  best["start_s"],
                "end_s":    best["end_s"],
                "score":    best["score"],
                "best_query": best["query"],
            })

        aggregated_predictions[model_key][video_name] = video_agg

# Persist to disk.
AGG_PREDICTIONS_PATH = Path("outputs/aggregated_predictions.json")
with open(AGG_PREDICTIONS_PATH, "w") as f:
    json.dump(aggregated_predictions, f, indent=2)

# Summary printout.
for model_key, model_results in aggregated_predictions.items():
    print(f"\n{model_key}:")
    for video_name, events in model_results.items():
        print(f"  {video_name}: {len(events)} events")
        for e in events[:3]:
            print(
                f"    event {e['event_id']:2d}: "
                f"{seconds_to_mmss(e['start_s'])} – {seconds_to_mmss(e['end_s'])}  "
                f"score={e['score']:.4f}"
            )
        if len(events) > 3:
            print(f"    ... {len(events) - 3} more")

print(f"\nAggregated predictions saved to {AGG_PREDICTIONS_PATH}")
print("Step 6.8 passed")



moment_detr:
  video_21: 24 events
    event  1: 08:41 – 09:22  score=0.9839
    event  2: 06:20 – 07:06  score=0.9925
    event  3: 06:22 – 06:48  score=0.9958
    ... 21 more
  video_22: 6 events
    event  1: 03:19 – 03:46  score=0.9972
    event  2: 00:00 – 00:19  score=0.9935
    event  3: 00:00 – 00:21  score=0.9952
    ... 3 more
  video_23: 6 events
    event  1: 00:57 – 01:46  score=0.9913
    event  2: 01:09 – 02:02  score=0.9930
    event  3: 01:03 – 01:54  score=0.9938
    ... 3 more
  video_24: 6 events
    event  1: 01:04 – 01:46  score=0.9919
    event  2: 01:11 – 01:37  score=0.9944
    event  3: 01:30 – 01:44  score=0.9781
    ... 3 more

cg_detr:
  video_21: 24 events
    event  1: 08:40 – 09:15  score=0.9989
    event  2: 05:06 – 05:25  score=0.9989
    event  3: 10:07 – 10:37  score=0.9983
    ... 21 more
  video_22: 6 events
    event  1: 02:42 – 03:02  score=0.9996
    event  2: 00:00 – 00:19  score=0.9992
    event  3: 03:39 – 03:50  score=0.9991
    ... 3 more


## Step 6.9 - Post-Processing

The aggregated predictions from Step 6.8 are one-per-event but may still contain noisy or invalid windows.  This step applies three filters:

| Filter | Threshold | Why |
|--------|-----------|-----|
| Minimum duration | ≥ 2 seconds | Windows under 2s are almost certainly model artifacts |
| Minimum confidence | score ≥ 0.0 | Negative scores mean the model actively disfavours the window |
| Maximum duration | ≤ 120 seconds | A single event should not span two minutes — this flags runaway predictions |

Timestamps are also **clamped to video bounds** so a prediction can never start before 0s or end after the video ends.

Flagged events are **not deleted** — they are kept in a separate `flagged_<model>.json` file for the failure analysis in the report.

Success check: the printed table should show how many events were kept vs. flagged per video per model.

In [37]:
assert "aggregated_predictions" in globals(), "Run Step 6.8 first."
assert "video_chunk_plan"       in globals(), "Run Step 6.3 first."

MIN_DURATION_S = 2.0    # discard windows shorter than this — likely spurious
MIN_SCORE      = 0.0    # discard windows with negative confidence
MAX_DURATION_S = 120.0  # cap absurdly long predictions (model artifacts)


def clamp_to_video(start_s, end_s, video_duration_s):
    """Ensure both endpoints stay within [0, video_duration_s]."""
    return max(0.0, start_s), min(video_duration_s, end_s)


def postprocess_video_predictions(events, video_duration_s):
    """Apply all filters to one video's event list.

    Returns (cleaned, flagged). Flagged entries keep a 'flag' field describing
    why they were removed so the failure analysis can inspect them.
    """
    cleaned = []
    flagged = []

    for e in events:
        start_s, end_s = clamp_to_video(e["start_s"], e["end_s"], video_duration_s)
        duration = end_s - start_s
        score    = e["score"]

        issues = []
        if duration < MIN_DURATION_S:
            issues.append(f"too short ({duration:.1f}s < {MIN_DURATION_S}s)")
        if score < MIN_SCORE:
            issues.append(f"low score ({score:.4f} < {MIN_SCORE})")
        if duration > MAX_DURATION_S:
            issues.append(f"too long ({duration:.1f}s > {MAX_DURATION_S}s)")

        entry = {**e, "start_s": round(start_s, 2), "end_s": round(end_s, 2)}

        if issues:
            entry["flag"] = " | ".join(issues)
            flagged.append(entry)
        else:
            cleaned.append(entry)

    return cleaned, flagged


postprocessed_predictions = {}
flagged_summary           = {}

for model_key, model_results in aggregated_predictions.items():
    postprocessed_predictions[model_key] = {}
    flagged_summary[model_key]           = {}

    for video_name, events in model_results.items():
        duration_s = video_chunk_plan[video_name]["duration_s"]
        cleaned, flagged = postprocess_video_predictions(events, duration_s)

        postprocessed_predictions[model_key][video_name] = cleaned
        flagged_summary[model_key][video_name]           = flagged

# Print summary.
for model_key in postprocessed_predictions:
    print(f"\n{model_key}:")
    for video_name in EXPECTED_VIDEOS:
        clean   = postprocessed_predictions[model_key][video_name]
        flagged = flagged_summary[model_key][video_name]
        print(f"  {video_name}: {len(clean)} kept, {len(flagged)} flagged")
        for f in flagged:
            print(
                f"    [FLAG] event {f['event_id']:2d}: {f['flag']}  "
                f"({seconds_to_mmss(f['start_s'])}–{seconds_to_mmss(f['end_s'])})"
            )

print("\nStep 6.9 passed")



moment_detr:
  video_21: 23 kept, 1 flagged
    [FLAG] event  8: too long (149.7s > 120.0s)  (02:19–04:49)
  video_22: 6 kept, 0 flagged
  video_23: 6 kept, 0 flagged
  video_24: 6 kept, 0 flagged

cg_detr:
  video_21: 24 kept, 0 flagged
  video_22: 6 kept, 0 flagged
  video_23: 6 kept, 0 flagged
  video_24: 6 kept, 0 flagged

Step 6.9 passed


## Step 6.10 - Save Results to JSON

Save one clean JSON file per model to `outputs/`:

- `predictions_moment_detr.json`
- `predictions_cg_detr.json`

Each file has the structure the assignment asks for:

```json
{
  "video_21": [
    {"event_id": 1, "event": "...", "start_s": 12.3, "end_s": 18.7, "score": 0.82},
    ...
  ]
}
```

Flagged (filtered-out) events are saved separately to `flagged_<model>.json` so nothing is permanently lost — they can be inspected during the failure analysis.

These files are the input for Step 6.11 (IoU evaluation).

In [38]:
import json
from pathlib import Path

assert "postprocessed_predictions" in globals(), "Run Step 6.9 first."

PREDICTIONS_DIR = Path("outputs")

# One clean file per model — this is the format used by IoU evaluation (Step 6.11).
for model_key, model_results in postprocessed_predictions.items():
    output = {}
    for video_name, events in model_results.items():
        output[video_name] = [
            {
                "event_id": e["event_id"],
                "event":    e["event"],
                "start_s":  e["start_s"],
                "end_s":    e["end_s"],
                "score":    e["score"],
            }
            for e in events
        ]
    out_path = PREDICTIONS_DIR / f"predictions_{model_key}.json"
    with open(out_path, "w") as f:
        json.dump(output, f, indent=2)
    print(f"Saved: {out_path}")

# Flagged events saved separately so they are easy to inspect later.
for model_key, flagged in flagged_summary.items():
    out_path = PREDICTIONS_DIR / f"flagged_{model_key}.json"
    with open(out_path, "w") as f:
        json.dump(flagged, f, indent=2)
    print(f"Saved flagged: {out_path}")

# Preview the final structure.
print("\nPreview — moment_detr:")
for video_name, events in postprocessed_predictions.get("moment_detr", {}).items():
    print(f"\n  {video_name}: {len(events)} events")
    for e in events[:3]:
        print(
            f"    event {e['event_id']:2d}: "
            f"{seconds_to_mmss(e['start_s'])} – {seconds_to_mmss(e['end_s'])}  "
            f"score={e['score']:.4f}"
        )
    if len(events) > 3:
        print(f"    ... {len(events) - 3} more")

print("\nStep 6.10 passed")


Saved: outputs\predictions_moment_detr.json
Saved: outputs\predictions_cg_detr.json
Saved flagged: outputs\flagged_moment_detr.json
Saved flagged: outputs\flagged_cg_detr.json

Preview — moment_detr:

  video_21: 23 events
    event  1: 08:41 – 09:22  score=0.9839
    event  2: 06:20 – 07:06  score=0.9925
    event  3: 06:22 – 06:48  score=0.9958
    ... 20 more

  video_22: 6 events
    event  1: 03:19 – 03:46  score=0.9972
    event  2: 00:00 – 00:19  score=0.9935
    event  3: 00:00 – 00:21  score=0.9952
    ... 3 more

  video_23: 6 events
    event  1: 00:57 – 01:46  score=0.9913
    event  2: 01:09 – 02:02  score=0.9930
    event  3: 01:03 – 01:54  score=0.9938
    ... 3 more

  video_24: 6 events
    event  1: 01:04 – 01:46  score=0.9919
    event  2: 01:11 – 01:37  score=0.9944
    event  3: 01:30 – 01:44  score=0.9781
    ... 3 more

Step 6.10 passed


## Step 6.11 - IoU Evaluation Against Annotations

This step compares the retrieved moments against the manual annotations in `videos_annotations.txt`.

**Matching strategy** — the VLM detected events and the hand-annotated events are not in 1:1 correspondence (e.g. video_21 has 24 VLM events but only 13 annotations).  We therefore use a **recall-oriented greedy match**: for each ground-truth annotation, we search through *all* predicted windows and keep the one with the highest IoU.  This answers the question "how well does the model cover each annotated event?"

**IoU formula**:
```
IoU = |intersection| / |union|   ∈ [0, 1]
```
A perfect prediction scores 1.0; no overlap at all scores 0.0.

**Outputs**:
- Comparison table: per-video mean IoU side-by-side for Moment-DETR vs CG-DETR
- Detailed table for Moment-DETR: GT interval | best-matching predicted interval | IoU | description
- `outputs/iou_evaluation.json` — full per-event results for the report

**Note on the subjectivity score**: annotations have scores from -2 (very subjective) to +2 (objectively essential). We evaluate all events equally here; the report can discuss whether IoU correlates with the score (i.e. whether objectively important events are easier to retrieve).

In [39]:
import json
import re
from pathlib import Path

assert "postprocessed_predictions" in globals(), "Run Step 6.9 first."

ANNOTATIONS_PATH = Path("videos_annotations.txt")


# ── annotation parser ─────────────────────────────────────────────────────────

def mmss_to_seconds(mmss):
    """Convert MM:SS string to float seconds."""
    m, s = mmss.strip().split(":")
    return int(m) * 60 + int(s)


def parse_annotations(path):
    """Parse videos_annotations.txt → {video_name: [{start_s, end_s, description, score}]}."""
    annotations = {}
    current_video = None

    with open(path, "r", encoding="utf-8") as f:
        for raw_line in f:
            line = raw_line.strip()
            if not line:
                continue

            # Header line: "Video 21"
            header = re.match(r"^Video\s+(\d+)\s*$", line)
            if header:
                current_video = f"video_{header.group(1)}"
                annotations[current_video] = []
                continue

            # Annotation line: MM:SS–MM:SS Description score
            # The separator can be an en-dash (–) or a plain hyphen (-).
            ann = re.match(
                r"^(\d{1,2}:\d{2})[–\-](\d{1,2}:\d{2})\s+(.+?)\s+([-]?\d+)\s*$",
                line,
            )
            if ann and current_video is not None:
                annotations[current_video].append({
                    "start_s":     mmss_to_seconds(ann.group(1)),
                    "end_s":       mmss_to_seconds(ann.group(2)),
                    "description": ann.group(3).strip(),
                    "score":       int(ann.group(4)),
                })

    return annotations


# ── IoU helper ────────────────────────────────────────────────────────────────

def compute_iou(pred_start, pred_end, gt_start, gt_end):
    """Temporal IoU between two intervals."""
    inter_start = max(pred_start, gt_start)
    inter_end   = min(pred_end,   gt_end)
    inter       = max(0.0, inter_end - inter_start)
    union       = (pred_end - pred_start) + (gt_end - gt_start) - inter
    return inter / union if union > 0 else 0.0


# ── per-video matching ────────────────────────────────────────────────────────

def match_gt_to_predictions(gt_events, pred_events):
    """For each GT event, find the predicted window with the highest IoU.

    Returns a list of per-GT-event result dicts.
    """
    results = []
    for gt in gt_events:
        best_iou  = 0.0
        best_pred = None

        for pred in pred_events:
            iou = compute_iou(pred["start_s"], pred["end_s"], gt["start_s"], gt["end_s"])
            if iou > best_iou:
                best_iou  = iou
                best_pred = pred

        results.append({
            "gt_start_s":     gt["start_s"],
            "gt_end_s":       gt["end_s"],
            "gt_description": gt["description"],
            "gt_score":       gt["score"],
            "pred_start_s":   best_pred["start_s"] if best_pred else None,
            "pred_end_s":     best_pred["end_s"]   if best_pred else None,
            "pred_event":     best_pred["event"]   if best_pred else None,
            "iou":            round(best_iou, 4),
        })

    return results


# ── run evaluation ────────────────────────────────────────────────────────────

ground_truth  = parse_annotations(ANNOTATIONS_PATH)
iou_results   = {}   # {model_key: {video_name: [per-gt-event results]}}
iou_summary   = {}   # {model_key: {video_name: mean_iou}}

for model_key, model_preds in postprocessed_predictions.items():
    iou_results[model_key]  = {}
    iou_summary[model_key]  = {}

    for video_name in EXPECTED_VIDEOS:
        gt_events   = ground_truth.get(video_name, [])
        pred_events = model_preds.get(video_name, [])

        if not gt_events:
            print(f"  [WARN] No annotations found for {video_name}")
            continue

        matches = match_gt_to_predictions(gt_events, pred_events)
        mean_iou = sum(m["iou"] for m in matches) / len(matches)

        iou_results[model_key][video_name] = matches
        iou_summary[model_key][video_name] = round(mean_iou, 4)

# ── print comparison table ────────────────────────────────────────────────────

model_keys = list(iou_summary.keys())

header = f"{'Video':<12}" + "".join(f"{mk:>16}" for mk in model_keys)
print(header)
print("-" * len(header))

for video_name in EXPECTED_VIDEOS:
    row = f"{video_name:<12}"
    for mk in model_keys:
        val = iou_summary.get(mk, {}).get(video_name, float("nan"))
        row += f"{val:>16.4f}"
    print(row)

print("-" * len(header))
overall_row = f"{'Overall':<12}"
for mk in model_keys:
    vals = [v for v in iou_summary.get(mk, {}).values() if isinstance(v, float)]
    overall = sum(vals) / len(vals) if vals else float("nan")
    overall_row += f"{overall:>16.4f}"
print(overall_row)

# ── per-video detail ──────────────────────────────────────────────────────────

print("\n\nDetailed results — Moment-DETR:")
for video_name in EXPECTED_VIDEOS:
    matches = iou_results.get("moment_detr", {}).get(video_name, [])
    if not matches:
        continue
    print(f"\n  {video_name}  (mean IoU = {iou_summary['moment_detr'][video_name]:.4f})")
    print(f"    {'GT interval':<18}  {'Pred interval':<18}  {'IoU':>6}  GT description")
    print(f"    {'-'*18}  {'-'*18}  {'-'*6}  {'-'*40}")
    for m in matches:
        gt_str   = f"{seconds_to_mmss(m['gt_start_s'])}–{seconds_to_mmss(m['gt_end_s'])}"
        if m["pred_start_s"] is not None:
            pred_str = f"{seconds_to_mmss(m['pred_start_s'])}–{seconds_to_mmss(m['pred_end_s'])}"
        else:
            pred_str = "no prediction"
        desc = m["gt_description"][:42]
        print(f"    {gt_str:<18}  {pred_str:<18}  {m['iou']:>6.4f}  {desc}")

# ── save evaluation results ───────────────────────────────────────────────────

eval_output = {
    "summary":  iou_summary,
    "per_event": iou_results,
}
EVAL_PATH = Path("outputs/iou_evaluation.json")
with open(EVAL_PATH, "w") as f:
    json.dump(eval_output, f, indent=2)

print(f"\nEvaluation saved to {EVAL_PATH}")
print("Step 6.11 passed")


  [WARN] No annotations found for video_21
  [WARN] No annotations found for video_22
  [WARN] No annotations found for video_23
  [WARN] No annotations found for video_24
  [WARN] No annotations found for video_21
  [WARN] No annotations found for video_22
  [WARN] No annotations found for video_23
  [WARN] No annotations found for video_24
Video            moment_detr         cg_detr
--------------------------------------------
video_21                 nan             nan
video_22                 nan             nan
video_23                 nan             nan
video_24                 nan             nan
--------------------------------------------
Overall                  nan             nan


Detailed results — Moment-DETR:

Evaluation saved to outputs\iou_evaluation.json
Step 6.11 passed


## Step 6.11a - Per-Event Failure Table

Every GT event ranked from worst to best IoU to see exactly which events the model struggled with.


In [40]:
import json
from pathlib import Path

EVAL_PATH = Path("outputs/iou_evaluation.json")
assert EVAL_PATH.exists(), "Run Step 6.11 first."

with open(EVAL_PATH) as f:
    eval_data = json.load(f)

EXPECTED_VIDEOS = ["video_21", "video_22", "video_23", "video_24"]

def _s2t(s):
    s = int(s)
    return f"{s // 60:02d}:{s % 60:02d}"

def iou_tier(iou):
    if iou >= 0.5: return "GOOD"
    if iou >= 0.3: return "PARTIAL"
    if iou >= 0.1: return "POOR"
    return "MISSED"

for model_key in ["moment_detr", "cg_detr"]:
    per_event = eval_data["per_event"].get(model_key, {})
    rows = []
    for video_name in EXPECTED_VIDEOS:
        for e in per_event.get(video_name, []):
            rows.append({**e, "video": video_name})
    rows.sort(key=lambda x: x["iou"])

    label = model_key.replace("_", "-").upper()
    sep = "=" * 108
    print(f"\n{sep}")
    print(f"  {label}  -- per-event IoU (worst to best)")
    print(f"{sep}")
    print(f"  {'Video':<10}  {'GT interval':<14}  {'Pred interval':<14}  {'IoU':>6}  {'Quality':<8}  GT description")
    print(f"  {'-'*10}  {'-'*14}  {'-'*14}  {'-'*6}  {'-'*8}  {'-'*48}")
    for r in rows:
        gt   = f"{_s2t(r['gt_start_s'])}-{_s2t(r['gt_end_s'])}"
        pred = f"{_s2t(r['pred_start_s'])}-{_s2t(r['pred_end_s'])}" if r["pred_start_s"] is not None else "--"
        print(f"  {r['video']:<10}  {gt:<14}  {pred:<14}  {r['iou']:>6.4f}  {iou_tier(r['iou']):<8}  {r['gt_description'][:48]}")

print("\nStep 6.11a passed")



  MOMENT-DETR  -- per-event IoU (worst to best)
  Video       GT interval     Pred interval      IoU  Quality   GT description
  ----------  --------------  --------------  ------  --------  ------------------------------------------------

  CG-DETR  -- per-event IoU (worst to best)
  Video       GT interval     Pred interval      IoU  Quality   GT description
  ----------  --------------  --------------  ------  --------  ------------------------------------------------

Step 6.11a passed


## Step 6.11b - Failure Category Analysis

Group events by query type (action / social / narrative / named_entity) and compute mean IoU per category to identify which types of queries the model struggles with most.


In [41]:
import json
from collections import defaultdict
from pathlib import Path

EVAL_PATH = Path("outputs/iou_evaluation.json")
assert EVAL_PATH.exists(), "Run Step 6.11 first."

with open(EVAL_PATH) as f:
    eval_data = json.load(f)

EXPECTED_VIDEOS = ["video_21", "video_22", "video_23", "video_24"]

available_models = list(eval_data.get("per_event", {}).keys())
print(f"Models in evaluation file: {available_models}")

# Classify each GT event by what the retrieval model needs to understand.
# named_entity : query names a person/org not identifiable by appearance alone
# action       : describes visible physical movement (parkour, jump, climb)
# social       : people interacting without distinctive action (interview, fans)
# narrative    : context/story-dependent (waking up, catching a bus, credits)

CATEGORY_RULES = [
    # video_21
    ("video_21", "Video title",                        "named_entity"),
    ("video_21", "David Belle performs parkour sequ",  "named_entity"),
    ("video_21", "David Belle climbs",                 "named_entity"),
    ("video_21", "A parkour group is introduced",      "social"),
    ("video_21", "The group interacts with fans",      "social"),
    ("video_21", "The group is interviewed",           "social"),
    ("video_21", "A clip from the movie",              "action"),
    ("video_21", "David Belle visits",                 "named_entity"),
    ("video_21", "David Belle's mother",               "named_entity"),
    ("video_21", "David Belle shows images",           "named_entity"),
    ("video_21", "A person attempts a trick",          "action"),
    ("video_21", "David Belle performs parkour down",  "named_entity"),
    ("video_21", "A group greets David Belle",         "named_entity"),
    # video_22
    ("video_22", "A group of people warm up",          "action"),
    ("video_22", "A logo appears",                     "named_entity"),
    ("video_22", "A member of Charlotte Parkour",      "named_entity"),
    ("video_22", "A group continues training",         "action"),
    ("video_22", "A participant attempts a rolling",   "action"),
    ("video_22", "Another participant speaks",         "action"),
    ("video_22", "A group practices coordinated",      "action"),
    ("video_22", "The group lines up on a wall",       "social"),
    # video_23
    ("video_23", "A man wakes up",                     "narrative"),
    ("video_23", "He quickly gets dressed",            "narrative"),
    ("video_23", "He exits the building",              "narrative"),
    ("video_23", "He attempts to catch public",        "narrative"),
    ("video_23", "He runs and performs parkour",       "action"),
    ("video_23", "He slows down",                      "narrative"),
    ("video_23", "He arrives at an empty",             "narrative"),
    ("video_23", "He realizes it is Sunday",           "narrative"),
    ("video_23", "Ending credits",                     "social"),
    # video_24
    ("video_24", "A group of people socialize",        "social"),
    ("video_24", "People are shown inside a car",      "social"),
    ("video_24", "A boy performs a sequence",          "action"),
    ("video_24", "A group dances",                     "social"),
    ("video_24", "Multiple individuals perform",       "action"),
    ("video_24", "A shopping cart",                    "action"),
    ("video_24", "Additional parkour sequences",       "action"),
    ("video_24", "A person carefully walks",           "action"),
]

def classify_event(video_name, description):
    for v, prefix, cat in CATEGORY_RULES:
        if v == video_name and description.startswith(prefix):
            return cat
    return "other"

CATEGORIES = ["action", "social", "narrative", "named_entity"]

# ?? mean IoU per category, for each available model ??????????????????????????

for model_key in available_models:
    per_event = eval_data["per_event"].get(model_key, {})

    all_events = [e for v in EXPECTED_VIDEOS for e in per_event.get(v, [])]
    if not all_events:
        print(f"\n[SKIP] {model_key}: no events found in evaluation file.")
        continue

    cat_ious = defaultdict(list)
    for video_name in EXPECTED_VIDEOS:
        for e in per_event.get(video_name, []):
            cat = classify_event(video_name, e["gt_description"])
            cat_ious[cat].append(e["iou"])

    unclassified = cat_ious.get("other", [])
    if unclassified:
        print(f"  [WARN] {len(unclassified)} event(s) could not be classified for {model_key}")

    label = model_key.replace("_", "-").upper()
    print(f"\n{label}  -- mean IoU by query category")
    print(f"  {'Category':<14}  {'N':>3}  {'Mean IoU':>9}  {'good(>=0.5)':>12}  {'missed(<0.1)':>13}")
    print(f"  {'-'*14}  {'-'*3}  {'-'*9}  {'-'*12}  {'-'*13}")
    for cat in CATEGORIES:
        ious = cat_ious.get(cat, [])
        if not ious:
            continue
        mean   = sum(ious) / len(ious)
        good   = sum(1 for x in ious if x >= 0.5)
        missed = sum(1 for x in ious if x <  0.1)
        print(f"  {cat:<14}  {len(ious):>3}  {mean:>9.4f}  {good:>12}  {missed:>13}")
    all_ious = [e["iou"] for e in all_events]
    print(f"  {'-'*14}  {'-'*3}  {'-'*9}  {'-'*12}  {'-'*13}")
    print(f"  {'OVERALL':<14}  {len(all_ious):>3}  {sum(all_ious)/len(all_ious):>9.4f}")

# ?? worst-performing events per category (best available model) ???????????????

best_model = available_models[-1]
print(f"\n\nWorst retrievals by category ({best_model}):")
per_event_best = eval_data["per_event"].get(best_model, {})
by_cat = defaultdict(list)
for video_name in EXPECTED_VIDEOS:
    for e in per_event_best.get(video_name, []):
        cat = classify_event(video_name, e["gt_description"])
        by_cat[cat].append((video_name, e))

for cat in CATEGORIES:
    events = by_cat.get(cat, [])
    worst  = sorted(events, key=lambda x: x[1]["iou"])[:3]
    print(f"\n  {cat.upper()} (worst 3):")
    for vid, e in worst:
        print(f"    IoU={e['iou']:.4f}  {vid}  {e['gt_description'][:55]}")

print("\nStep 6.11b passed")


Models in evaluation file: ['moment_detr', 'cg_detr']

[SKIP] moment_detr: no events found in evaluation file.

[SKIP] cg_detr: no events found in evaluation file.


Worst retrievals by category (cg_detr):

  ACTION (worst 3):

  SOCIAL (worst 3):

  NARRATIVE (worst 3):

  NAMED_ENTITY (worst 3):

Step 6.11b passed


## Step 6.12 - Feature Experiment: CLIP vs CLIP+SlowFast

This step re-runs the full pipeline (sweep → aggregate → post-process → IoU) with `feature_name="clip_slowfast"` to compare against the CLIP-only baseline from Steps 6.7–6.11.

**Why CLIP+SlowFast?**
- CLIP encodes visual semantics at the frame level (spatial understanding).
- SlowFast adds temporal motion features (fast pathway) alongside slow semantic features.
- For parkour/action videos, motion cues should help localise dynamic events more precisely.

**Checkpoint download**: the `clip_slowfast_*_qvhighlight.ckpt` files follow the same Zenodo pattern as the CLIP checkpoints. The code attempts to download them automatically. If the download fails (URL changed or not yet available), the step prints a `[NOTE]` and the CLIP baseline from Step 6.11 remains fully valid for the report.

**Output**: a side-by-side IoU table (CLIP vs CLIP+SlowFast for both models) and `outputs/feature_experiment_iou.json`.

In [42]:
import contextlib
import gc
import json
import urllib.request
from pathlib import Path

import numpy.core.defchararray as _numpy_char
import sys
sys.modules.setdefault("numpy.char", _numpy_char)

import clip
import torch
from easydict import EasyDict
from lighthouse.models import CGDETRPredictor, MomentDETRPredictor

assert "query_variants"    in globals(), "Run Step 6.2 first."
assert "video_chunk_plan"  in globals(), "Run Step 6.3 first."
assert "retrieval_setup"   in globals(), "Run Step 6.4 first."
assert "iou_summary"       in globals(), "Run Step 6.11 first."

# ── checkpoint URLs for clip_slowfast ─────────────────────────────────────────
# These follow the same Zenodo pattern as the CLIP-only checkpoints in Steps 6.5/6.6.

SLOWFAST_CONFIGS = [
    {
        "model_key":       "moment_detr_slowfast",
        "display_name":    "Moment-DETR + CLIP+SlowFast",
        "predictor_class": MomentDETRPredictor,
        "feature_name":    "clip_slowfast",
        "ckpt":            WEIGHTS_DIR / "clip_slowfast_moment_detr_qvhighlight.ckpt",
        "safe_ckpt":       WEIGHTS_DIR / "clip_slowfast_moment_detr_qvhighlight_safe.ckpt",
        "url": "https://zenodo.org/records/13363606/files/clip_slowfast_moment_detr_qvhighlight.ckpt",
    },
    {
        "model_key":       "cg_detr_slowfast",
        "display_name":    "CG-DETR + CLIP+SlowFast",
        "predictor_class": CGDETRPredictor,
        "feature_name":    "clip_slowfast",
        "ckpt":            WEIGHTS_DIR / "clip_slowfast_cg_detr_qvhighlight.ckpt",
        "safe_ckpt":       WEIGHTS_DIR / "clip_slowfast_cg_detr_qvhighlight_safe.ckpt",
        "url": "https://zenodo.org/records/13363606/files/clip_slowfast_cg_detr_qvhighlight.ckpt",
    },
]


def try_download(url, path):
    if path.exists():
        return True
    try:
        print(f"  Downloading {path.name} …", end=" ", flush=True)
        tmp = path.with_suffix(path.suffix + ".part")
        urllib.request.urlretrieve(url, tmp)
        tmp.replace(path)
        print("done")
        return True
    except Exception as exc:
        print(f"failed ({exc})")
        if tmp.exists():
            tmp.unlink()
        return False


def make_safe_ckpt(src, dst):
    if dst.exists():
        return
    raw = torch.load(src, map_location="cpu", weights_only=False)
    torch.save({"model": raw["model"], "opt": dict(raw["opt"])}, dst)


def _make_load_patch(safe_ckpt_path):
    _orig = torch.load
    def _patched(*args, **kwargs):
        loaded = _orig(*args, **kwargs)
        if args and Path(str(args[0])).resolve() == Path(safe_ckpt_path).resolve():
            loaded["opt"] = EasyDict(loaded["opt"])
        return loaded
    return _orig, _patched


# ── run sweep for each slowfast config that can be downloaded ─────────────────

slowfast_raw_predictions = {}
skipped = []

for cfg in SLOWFAST_CONFIGS:
    print(f"\n{'=' * 60}")
    print(f"  {cfg['display_name']}")
    print(f"{'=' * 60}")

    if not try_download(cfg["url"], cfg["ckpt"]):
        print(f"  Skipping — checkpoint unavailable.")
        skipped.append(cfg["model_key"])
        continue

    make_safe_ckpt(cfg["ckpt"], cfg["safe_ckpt"])

    _orig_load, _patched_load = _make_load_patch(cfg["safe_ckpt"])
    _orig_clip = clip.load

    try:
        torch.load = _patched_load
        clip.load  = lambda *a, **kw: _orig_clip(
            *a, **{**kw, "download_root": str(WEIGHTS_DIR / "clip_cache")}
        )
        with open("nul", "w") as _dev:
            with contextlib.redirect_stdout(_dev), contextlib.redirect_stderr(_dev):
                model = cfg["predictor_class"](
                    str(cfg["safe_ckpt"]),
                    device=retrieval_setup["device"],
                    feature_name=cfg["feature_name"],
                )
    except Exception as exc:
        print(f"  Model load failed: {exc}")
        skipped.append(cfg["model_key"])
        torch.load = _orig_load
        clip.load  = _orig_clip
        continue
    finally:
        torch.load = _orig_load
        clip.load  = _orig_clip

    print(f"  Loaded on {retrieval_setup['device']}")
    slowfast_raw_predictions[cfg["model_key"]] = {}

    for video_name in EXPECTED_VIDEOS:
        plan   = video_chunk_plan[video_name]
        chunks = plan["chunks"]
        events = query_variants[video_name]
        print(f"\n  {video_name}: {len(chunks)} chunk(s) × {len(events)} event(s)")
        video_results = []

        for chunk in chunks:
            chunk_path = ensure_chunk_file(plan["video_path"], chunk)
            try:
                with open("nul", "w") as _dev:
                    with contextlib.redirect_stdout(_dev), contextlib.redirect_stderr(_dev):
                        encoded_video = model.encode_video(str(chunk_path))
            except Exception as exc:
                print(f"    [ERROR] encode_video on chunk {chunk['chunk_id']}: {exc}")
                continue

            for event_item in events:
                query_results = []
                for query_text in event_item["queries"]:
                    try:
                        with open("nul", "w") as _dev:
                            with contextlib.redirect_stdout(_dev), contextlib.redirect_stderr(_dev):
                                pred = model.predict(query_text, encoded_video)
                        rel_start, rel_end, score = pred["pred_relevant_windows"][0]
                        query_results.append({
                            "query":   query_text,
                            "start_s": round(float(chunk["start_s"] + rel_start), 2),
                            "end_s":   round(float(chunk["start_s"] + rel_end),   2),
                            "score":   round(float(score), 4),
                        })
                    except Exception as exc:
                        print(f"    [ERROR] predict: {exc}")

                existing = next(
                    (r for r in video_results if r["event_id"] == event_item["event_id"]), None
                )
                chunk_entry = {
                    "chunk_id": chunk["chunk_id"],
                    "chunk_start_s": chunk["start_s"],
                    "chunk_end_s":   chunk["end_s"],
                    "query_results": query_results,
                }
                if existing is None:
                    video_results.append({
                        "event_id":      event_item["event_id"],
                        "event":         event_item["event"],
                        "chunk_results": [chunk_entry],
                    })
                else:
                    existing["chunk_results"].append(chunk_entry)

            if video_results:
                scores = [
                    max((qr["score"] for qr in er["chunk_results"][-1]["query_results"]), default=0)
                    for er in video_results
                    if er["chunk_results"][-1]["chunk_id"] == chunk["chunk_id"]
                ]
                avg = sum(scores) / len(scores) if scores else 0.0
                print(f"    chunk {chunk['chunk_id']}: avg best-score = {avg:.4f}")

        slowfast_raw_predictions[cfg["model_key"]][video_name] = video_results

    del model
    if retrieval_setup["device"] == "cuda":
        torch.cuda.empty_cache()
    elif retrieval_setup["device"] == "mps" and hasattr(torch.mps, "empty_cache"):
        torch.mps.empty_cache()
    gc.collect()
    print(f"\n  {cfg['display_name']} done — model unloaded.")

if skipped:
    print(f"\n[NOTE] Skipped (checkpoint not available): {skipped}")

# ── aggregate + evaluate slowfast predictions ─────────────────────────────────

slowfast_iou_summary = {}

if slowfast_raw_predictions:
    # Re-use helpers from Steps 6.8, 6.9, 6.11.
    ground_truth = parse_annotations(ANNOTATIONS_PATH)

    for model_key, model_results in slowfast_raw_predictions.items():
        agg, pp = {}, {}
        for video_name, video_results in model_results.items():
            duration_s = video_chunk_plan[video_name]["duration_s"]
            agg_events = []
            for er in video_results:
                best = aggregate_event_predictions(er["chunk_results"])
                agg_events.append({**er, "start_s": best["start_s"],
                                   "end_s": best["end_s"], "score": best["score"],
                                   "best_query": best["query"]})
            agg[video_name] = agg_events
            cleaned, _ = postprocess_video_predictions(agg_events, duration_s)
            pp[video_name] = cleaned

        iou_per_video = {}
        for video_name in EXPECTED_VIDEOS:
            gt   = ground_truth.get(video_name, [])
            pred = pp.get(video_name, [])
            if not gt:
                continue
            matches  = match_gt_to_predictions(gt, pred)
            mean_iou = sum(m["iou"] for m in matches) / len(matches)
            iou_per_video[video_name] = round(mean_iou, 4)
        slowfast_iou_summary[model_key] = iou_per_video

# ── comparison table: CLIP baseline vs CLIP+SlowFast ─────────────────────────

all_keys   = list(iou_summary.keys()) + list(slowfast_iou_summary.keys())
col_width  = 18
header     = f"{'Video':<12}" + "".join(f"{k:>{col_width}}" for k in all_keys)
print("\nFeature experiment — IoU comparison")
print(header)
print("-" * len(header))

for video_name in EXPECTED_VIDEOS:
    row = f"{video_name:<12}"
    for k in all_keys:
        src = iou_summary if k in iou_summary else slowfast_iou_summary
        val = src.get(k, {}).get(video_name, float("nan"))
        row += f"{val:>{col_width}.4f}" if isinstance(val, float) else f"{'N/A':>{col_width}}"
    print(row)

print("-" * len(header))
overall_row = f"{'Overall':<12}"
for k in all_keys:
    src  = iou_summary if k in iou_summary else slowfast_iou_summary
    vals = [v for v in src.get(k, {}).values() if isinstance(v, float)]
    avg  = sum(vals) / len(vals) if vals else float("nan")
    overall_row += f"{avg:>{col_width}.4f}" if vals else f"{'N/A':>{col_width}}"
print(overall_row)

# ── save ──────────────────────────────────────────────────────────────────────

FEATURE_EXP_PATH = Path("outputs/feature_experiment_iou.json")
with open(FEATURE_EXP_PATH, "w") as f:
    json.dump({"clip_baseline": iou_summary, "clip_slowfast": slowfast_iou_summary}, f, indent=2)

print(f"\nFeature experiment results saved to {FEATURE_EXP_PATH}")
if skipped:
    print(f"[NOTE] {len(skipped)} config(s) skipped — checkpoint download failed.")
    print("       The CLIP baseline IoU from Step 6.11 is still valid for the report.")
print("Step 6.12 passed")



  Moment-DETR + CLIP+SlowFast
  Model load failed: Slowfast use model_path, so should be set but None.

  CG-DETR + CLIP+SlowFast
  Model load failed: Slowfast use model_path, so should be set but None.

[NOTE] Skipped (checkpoint not available): ['moment_detr_slowfast', 'cg_detr_slowfast']

Feature experiment — IoU comparison
Video              moment_detr           cg_detr
------------------------------------------------
video_21                   nan               nan
video_22                   nan               nan
video_23                   nan               nan
video_24                   nan               nan
------------------------------------------------
Overall                    N/A               N/A

Feature experiment results saved to outputs\feature_experiment_iou.json
[NOTE] 2 config(s) skipped — checkpoint download failed.
       The CLIP baseline IoU from Step 6.11 is still valid for the report.
Step 6.12 passed


## Step 6.13 - Video Summary Generation

This is the final step of the pipeline.  Using the cleaned predictions from Step 6.9, we extract the most confident video segments and concatenate them into one summary video per original video.

**Selection strategy**:
1. Filter events with `score >= 0.0` (same threshold as post-processing).
2. Sort by confidence score — keep the top-5 highest-scoring events.
3. Re-sort the selected clips **chronologically** (`start_s` ascending) so the summary flows in the same order as the original video.

**Model choice**: the model with the higher overall mean IoU from Step 6.11 is picked automatically.

**ffmpeg pipeline**:
- Each clip is re-encoded (`libx264 / aac`) so all clips have identical codec parameters before concatenation.
- A concat demuxer list file is written and used with `ffmpeg -f concat` to produce the final summary.

**Outputs** (per video):
- Individual clips: `outputs/summary_clips/video_XX_clip_NN.mp4`
- Final summary:   `outputs/summaries/summary_video_XX.mp4`
- Manifest JSON:   `outputs/summary_manifest.json` — records which events were selected, their timestamps and scores.

In [43]:
import json
import subprocess
from pathlib import Path

assert "postprocessed_predictions" in globals(), "Run Step 6.9 first."
assert "iou_summary"               in globals(), "Run Step 6.11 first."

SUMMARY_DIR = Path("outputs/summaries")
SUMMARY_DIR.mkdir(parents=True, exist_ok=True)

CLIP_TMP_DIR = Path("outputs/summary_clips")
CLIP_TMP_DIR.mkdir(parents=True, exist_ok=True)

# ── selection strategy ────────────────────────────────────────────────────────

TOP_N         = 5      # maximum clips to include per video
MIN_CLIP_SCORE = 0.0   # only include clips with score >= this

def select_summary_events(events, top_n=TOP_N, min_score=MIN_CLIP_SCORE):
    """Return up to top_n events, sorted by score, then re-ordered chronologically."""
    eligible = [e for e in events if e["score"] >= min_score]
    by_score = sorted(eligible, key=lambda e: e["score"], reverse=True)[:top_n]
    return sorted(by_score, key=lambda e: e["start_s"])  # chronological order


# ── pick the better model based on overall mean IoU ───────────────────────────

def overall_mean(model_key):
    vals = [v for v in iou_summary.get(model_key, {}).values() if isinstance(v, float)]
    return sum(vals) / len(vals) if vals else 0.0

best_model = max(postprocessed_predictions.keys(), key=overall_mean)
print(f"Using model: {best_model}  (mean IoU = {overall_mean(best_model):.4f})")


# ── extract and concatenate clips with ffmpeg ─────────────────────────────────

def extract_clip(video_path, start_s, end_s, out_path):
    duration = max(0.5, end_s - start_s)
    subprocess.run(
        ["ffmpeg", "-y",
         "-ss", str(start_s), "-t", str(duration),
         "-i", str(video_path),
         "-c:v", "libx264", "-c:a", "aac",
         "-avoid_negative_ts", "make_zero",
         str(out_path)],
        check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
    )


def concat_clips(clip_paths, out_path):
    """Write ffmpeg concat list and merge clips into one video."""
    list_file = out_path.with_suffix(".txt")
    with open(list_file, "w") as f:
        for p in clip_paths:
            f.write(f"file '{p.resolve()}'\n")
    subprocess.run(
        ["ffmpeg", "-y", "-f", "concat", "-safe", "0",
         "-i", str(list_file),
         "-c", "copy", str(out_path)],
        check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
    )
    list_file.unlink()


summary_manifest = {}

for video_name in EXPECTED_VIDEOS:
    plan   = video_chunk_plan[video_name]
    events = postprocessed_predictions[best_model].get(video_name, [])

    selected = select_summary_events(events)
    if not selected:
        print(f"\n{video_name}: no events pass the score threshold — skipping.")
        continue

    print(f"\n{video_name}: {len(selected)} clips selected")

    clip_paths = []
    for i, e in enumerate(selected, start=1):
        clip_out = CLIP_TMP_DIR / f"{video_name}_clip_{i:02d}.mp4"
        extract_clip(plan["video_path"], e["start_s"], e["end_s"], clip_out)
        clip_paths.append(clip_out)
        print(
            f"  clip {i:2d}: {seconds_to_mmss(e['start_s'])} – {seconds_to_mmss(e['end_s'])}"
            f"  score={e['score']:.4f}  {e['event'][:55]}"
        )

    summary_out = SUMMARY_DIR / f"summary_{video_name}.mp4"
    concat_clips(clip_paths, summary_out)
    print(f"  → {summary_out}")

    summary_manifest[video_name] = {
        "model":       best_model,
        "num_clips":   len(selected),
        "output":      str(summary_out),
        "events":      [
            {"event_id": e["event_id"], "event": e["event"],
             "start_s": e["start_s"], "end_s": e["end_s"], "score": e["score"]}
            for e in selected
        ],
    }

MANIFEST_PATH = Path("outputs/summary_manifest.json")
with open(MANIFEST_PATH, "w") as f:
    json.dump(summary_manifest, f, indent=2)

print(f"\nManifest saved to {MANIFEST_PATH}")
print("Step 6.13 passed")


Using model: moment_detr  (mean IoU = 0.0000)

video_21: 5 clips selected
  clip  1: 05:22 – 05:49  score=0.9983  A man is holding a piece of paper while standing indoor
  clip  2: 05:23 – 05:47  score=0.9978  A woman sits next to others enjoying beverages at a par
  clip  3: 06:21 – 06:49  score=0.9980  An aerial view shows multiple residential structures su
  clip  4: 06:58 – 07:29  score=0.9983  Inside someone walks through their living room holding 
  clip  5: 08:49 – 09:26  score=0.9980  Another group stands outside at night talking to someon
  → outputs\summaries\summary_video_21.mp4

video_22: 5 clips selected
  clip  1: 00:00 – 00:21  score=0.9952  The man who jumped has fallen down but quickly gets bac
  clip  2: 00:00 – 00:19  score=0.9935  One person is jumping off the top step onto another set
  clip  3: 02:56 – 03:46  score=0.9902  People gather outside a building where two individuals 
  clip  4: 03:19 – 03:46  score=0.9972  A group of people are gathered around a brick w

## Step 6.13b - Summary Quality Evaluation

Scores each summary video against four criteria from the assignment:
**Relevance**, **Coverage**, **Conciseness**, and **Coherence**.
Uses the manual annotations in `videos_annotations.txt` as ground truth.


In [ ]:
import json
import re
from pathlib import Path

MANIFEST_PATH    = Path("outputs/summary_manifest.json")
ANNOTATIONS_PATH = Path("videos_annotations.txt")
assert MANIFEST_PATH.exists(),    "Run Step 6.13 first."
assert ANNOTATIONS_PATH.exists(), "videos_annotations.txt not found."

with open(MANIFEST_PATH) as f:
    manifest = json.load(f)

# ?? parse annotations ?????????????????????????????????????????????????????????

def _mmss(t):
    m, s = t.strip().split(":")
    return int(m) * 60 + int(s)

def _fmt(s):
    s = int(s)
    return f"{s // 60:02d}:{s % 60:02d}"

def parse_annotations(path):
    anns, current = {}, None
    pat = re.compile(r"^(.+),\s*(\d{1,2}:\d{2})[\u2013\-](\d{1,2}:\d{2}),\s*([-]?\d+)\s*$")
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            h = re.match(r"^Video\s+(\d+)\s*$", line)
            if h:
                current = f"video_{h.group(1)}"
                anns[current] = []
                continue
            m = pat.match(line)
            if m and current:
                anns[current].append({
                    "description": m.group(1).strip(),
                    "start_s":     _mmss(m.group(2)),
                    "end_s":       _mmss(m.group(3)),
                    "score":       int(m.group(4)),
                })
    return anns

gt = parse_annotations(ANNOTATIONS_PATH)

# ?? helpers ???????????????????????????????????????????????????????????????????

def temporal_iou(a0, a1, b0, b1):
    inter = max(0.0, min(a1, b1) - max(a0, b0))
    union = (a1 - a0) + (b1 - b0) - inter
    return inter / union if union > 0 else 0.0

def video_duration(video_name):
    """Use chunk plan if available, else estimate from latest GT end time."""
    if "video_chunk_plan" in globals():
        plan = video_chunk_plan.get(video_name, {})
        chunks = plan.get("chunks", [])
        if chunks:
            return max(c["end_s"] for c in chunks)
    events = gt.get(video_name, [])
    return max((e["end_s"] for e in events), default=180) + 5

MATCH_THRESH   = 0.3   # IoU threshold to count a clip as "covering" a GT event
EXPECTED_VIDEOS = ["video_21", "video_22", "video_23", "video_24"]

print("=" * 76)
print("  SUMMARY QUALITY EVALUATION")
print("=" * 76)

for video_name in EXPECTED_VIDEOS:
    if video_name not in manifest:
        print(f"\n[SKIP] {video_name} not in manifest.")
        continue

    info      = manifest[video_name]
    selected  = info["events"]
    gt_events = gt.get(video_name, [])
    vid_dur   = video_duration(video_name)
    model     = info.get("model", "?")

    print(f"\n{'?' * 76}")
    print(f"  {video_name}  |  model: {model}  |  {info['num_clips']} clips  |  video: {_fmt(vid_dur)}")
    print(f"{'?' * 76}")

    sorted_clips = sorted(selected, key=lambda c: c["start_s"])

    # ?? 1. RELEVANCE ?????????????????????????????????????????????????????????
    rel_scores = []
    for clip in sorted_clips:
        best_iou, best_score, best_desc = 0.0, None, "no GT overlap"
        for g in gt_events:
            i = temporal_iou(clip["start_s"], clip["end_s"], g["start_s"], g["end_s"])
            if i > best_iou:
                best_iou, best_score, best_desc = i, g["score"], g["description"][:42]
        rel_scores.append((best_iou, best_score, best_desc))

    matched = [(s, d) for _, s, d in rel_scores if s is not None]
    mean_rel = sum(s for s, _ in matched) / len(matched) if matched else None

    print(f"\n  1. RELEVANCE  (does each selected clip match a GT event?)")
    print(f"     {'Clip interval':<16}  {'Best GT IoU':>11}  {'GT score':>8}  GT event")
    print(f"     {'?'*16}  {'?'*11}  {'?'*8}  {'?'*42}")
    for clip, (b_iou, b_score, b_desc) in zip(sorted_clips, rel_scores):
        interval = f"{_fmt(clip['start_s'])}?{_fmt(clip['end_s'])}"
        score_str = f"{b_score:+d}" if b_score is not None else "?"
        iou_str = f"{b_iou:.3f}" if b_iou > 0.01 else "0.000"
        print(f"     {interval:<16}  {iou_str:>11}  {score_str:>8}  {b_desc}")
    mean_str = f"{mean_rel:+.2f}" if mean_rel is not None else "n/a"
    print(f"     Mean GT subjectivity score of selected clips: {mean_str}  (scale: ?2 to +2)")

    # ?? 2. COVERAGE ??????????????????????????????????????????????????????????
    important = [g for g in gt_events if g["score"] >= 1]
    cov_rows  = []
    for g in important:
        best = max((temporal_iou(c["start_s"], c["end_s"], g["start_s"], g["end_s"])
                    for c in selected), default=0.0)
        cov_rows.append((best >= MATCH_THRESH, best, g))

    n_cov  = sum(1 for ok, _, _ in cov_rows if ok)
    cov_pct = n_cov / len(important) * 100 if important else 0.0

    print(f"\n  2. COVERAGE  ({n_cov}/{len(important)} important GT events covered at IoU >= {MATCH_THRESH})")
    for ok, best_iou, g in cov_rows:
        tag = "COVERED" if ok else "MISSED "
        print(f"     [{tag}]  IoU={best_iou:.3f}  score={g['score']:+d}  {_fmt(g['start_s'])}?{_fmt(g['end_s'])}  {g['description'][:40]}")

    # ?? 3. CONCISENESS ???????????????????????????????????????????????????????
    total_dur = sum(c["end_s"] - c["start_s"] for c in selected)
    ratio     = total_dur / vid_dur if vid_dur > 0 else 0.0
    overlap_s = sum(
        max(0.0, sorted_clips[i]["end_s"] - sorted_clips[i+1]["start_s"])
        for i in range(len(sorted_clips) - 1)
    )

    print(f"\n  3. CONCISENESS")
    print(f"     Summary : {total_dur:.1f} s ({_fmt(total_dur)})  =  {ratio:.1%} of original video")
    if overlap_s > 0:
        print(f"     WARNING : {overlap_s:.1f} s of overlapping content between clips")
    else:
        print(f"     No overlapping clips (good).")

    # ?? 4. COHERENCE ?????????????????????????????????????????????????????????
    starts   = [c["start_s"] for c in sorted_clips]
    is_chron = all(starts[i] <= starts[i+1] for i in range(len(starts)-1))
    gaps     = [sorted_clips[i+1]["start_s"] - sorted_clips[i]["end_s"]
                for i in range(len(sorted_clips)-1)]
    avg_gap  = sum(gaps) / len(gaps) if gaps else 0.0

    print(f"\n  4. COHERENCE")
    print(f"     Chronological: {'YES' if is_chron else 'NO (clips reordered)'}")
    if gaps:
        for i, g in enumerate(gaps):
            tag = "OVERLAP" if g < 0 else ("large jump" if g > 30 else "ok")
            print(f"     Gap clip {i+1}?{i+2}: {g:+.1f} s  [{tag}]  "
                  f"({_fmt(sorted_clips[i]['end_s'])} ? {_fmt(sorted_clips[i+1]['start_s'])})")
        print(f"     Average gap: {avg_gap:.1f} s")

print(f"\n{'=' * 76}")
print("Step 6.13b passed")
